# Calculation of occupational-level green/brown shares based on ESCO and ONET data
Felix Zaussinger | 17.05.2022

## Core Analysis Goal(s)
1. Calculate g/b shares based on ESCO skills data
2. Collect g/b shares based on various ONET data at different granularity levels
3. Combine into a central data set
4. Derive a final data set by using both ESCO and ONET data to triangulate green and brown occupations

**Overview of data sets**

1 ONET: Vona 2018 (Greenness), SOC 8-digit matched via ONET-ESCO crosswalk [CHECK]
2 ONET: GTP 2011 (Greenness), SOC 8-digit matched via ONET-ESCO crosswalk [CHECK]
3 ONET: Vona 2019 (Greenness), SOC 6-digit matched via IBS SOC-ISCO crosswalk
4 ONET: JRC/Consoli (Greenness), CP-2011 5-digit matched via CP2011-ESCO crosswalk
5 ESCO: ESCO 2022 (Greenness) [CHECK]

## Key Insight(s)
1.
2.
 3.

In [1]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from src import utils
import mapping_career_causeways

# OPTIONAL: Load the "autoreload" extension so that code can change
%load_ext autoreload

# OPTIONAL: always reload modules so that as you change code in src, it gets loaded
%autoreload 2

# Settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sns.set_context("poster")
sns.set(rc={'figure.figsize': (16, 9.)})
sns.set_style("ticks")

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

# load paths
useful_paths = utils.UsefulPaths()

**Initialise main objects**

In [95]:
from src.data.framework import Esco, Onet, Classifications, Crosswalks

esco = Esco()
onet = Onet()
classifications = Classifications()
crosswalks = Crosswalks()

**1) Combine ESCO skills metadata**

In [31]:
var_sel_skills = [
    'conceptType', 'conceptUri', 'skillType', 'reuseLevel', 'preferredLabel',
    'skill_green_esco', 'skill_brown_esco', 'skill_neutral_esco', 'skill_classification_esco',
    'skill_green_eth', 'skill_brown_eth', 'machineScore_green', 'machineScore_brown',
]

smd = esco.combine_skills_metadata(override=True, variable_selection=var_sel_skills)
smd.head()

C:\Users\fzaussinger\Miniconda3\envs\re4gt\lib\site-packages\openpyxl\worksheet\_reader.py:312: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
C:\Users\fzaussinger\Miniconda3\envs\re4gt\lib\site-packages\openpyxl\worksheet\_reader.py:312: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,conceptType,conceptUri,skillType,reuseLevel,preferredLabel,skill_green_esco,skill_brown_esco,skill_neutral_esco,skill_classification_esco,skill_green_eth,skill_brown_eth,machineScore_green,machineScore_brown
0,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0005c151-5b5a...,skill/competence,sector-specific,manage musical staff,False,False,True,neutral,NaN,NaN,NaN,NaN
1,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00064735-8fad...,skill/competence,occupation-specific,supervise correctional procedures,False,False,True,neutral,NaN,NaN,NaN,NaN
2,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/000709ed-2be5...,skill/competence,sector-specific,apply anti-oppressive practices,False,False,True,neutral,NaN,NaN,NaN,NaN
3,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0007bdc2-dd15...,skill/competence,sector-specific,control compliance of railway vehicles regulat...,False,False,True,neutral,NaN,NaN,NaN,NaN
4,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00090cc1-1f27...,skill/competence,cross-sector,identify available services,False,False,True,neutral,NaN,NaN,NaN,NaN


**2) Calculate GBN occupation shares based on ESCO skill classifications**

In [40]:
# based on essential and optional skills, unweighted
esco_gbn_shares_all = esco.calc_gbn_shares_skill_based(skills_metadata=smd, drop_counts=True, essential_only=False)

# based on essential skills only (similar to core greenness in Vona et al. 2019)
esco_gbn_shares_ess = esco.calc_gbn_shares_skill_based(skills_metadata=smd, drop_counts=True, essential_only=True)

T:\Documents\Projects\04_jrc_green-skills-regional\03_data-analysis\re4gt\src\data\framework.py:1361: RuntimeWarning: invalid value encountered in true_divide
  occ_share = n_gbn_specific_skills / n_total_specific_skills


In [43]:
esco_gbn_shares_ess.columns

Index(['conceptUri', 'share_green_esco', 'share_brown_esco',
       'share_neutral_esco', 'gbn_classification_esco'],
      dtype='object')

**3) Combine ESCO-level occupation metadata**

In [116]:
var_sel_occs = [
    "conceptUri", "preferredLabel", "description", "isco_level_4", "isco_level_3", "isco_level_2", "isco_level_1", "isco_label_4", "isco_label_3", "isco_label_2", "isco_label_1",
    'share_green_esco', 'share_brown_esco', 'share_neutral_esco', 'gbn_classification_esco',
    'share_green_esco_ess', 'share_brown_esco_ess', 'share_neutral_esco_ess', 'gbn_classification_esco_ess',
]

omd = esco.combine_occupation_metadata(skills_metadata=smd, variable_selection=var_sel_occs)

T:\Documents\Projects\04_jrc_green-skills-regional\03_data-analysis\re4gt\src\data\framework.py:1347: RuntimeWarning: invalid value encountered in true_divide
  # TODO: remove [:1] once ETH skill classification is finished


**4) Join 8-digit ONET-level occupation metadata to Nesta crosswalk**

In [49]:
# read Nesta crosswalk
cw_onet_esco = crosswalks.onet_esco_mcc_full
cw_onet_esco

,onet_code,title,share_green_vona2018,total_spec_tasks_vona2018,green_spec_tasks_vona2018
0,11-1011.03,Chief Sustainability Officers,1.000000,18,18
1,11-1021.00,General and Operations Managers,0.055556,18,1
2,11-2021.00,Marketing Managers,0.200000,20,4
3,11-3051.02,Geothermal Production Managers,1.000000,17,17
4,11-3051.04,Biomass Power Plant Managers,1.000000,18,18
5,11-3071.01,Transportation Managers,0.178571,28,5
6,11-3071.02,Storage and Distribution Managers,0.233333,30,7
7,11-3071.03,Logistics Managers,0.300000,30,9
8,11-9021.00,Construction Managers,0.280000,25,7
9,11-9041.00,Architectural and Engineering Managers,0.190476,21,4


Join Vona 2018 data to crosswalk at ONET 8-digit level

In [71]:
cw_onet_esco_merged = pd.merge(
    cw_onet_esco,
    onet.green_occupations_vona2018[["onet_code", "share_green_vona2018"]],
    on="onet_code",
    how="left"
)

# clean
cw_onet_esco_merged

,id,concept_uri,preferred_label,isco_level_4,onet_code,onet_occupation,onet_code_6d,share_green_vona2018
0,0,http://data.europa.eu/esco/occupation/00030d09...,technical director,2166,27-1011.00,art directors,27-1011,NaN
1,1,http://data.europa.eu/esco/occupation/000e93a3...,metal drawing machine operator,8121,51-4021.00,"extruding and drawing machine setters, operato...",51-4021,NaN
2,2,http://data.europa.eu/esco/occupation/0019b951...,precision device inspector,7543,51-9061.00,"inspectors, testers, sorters, samplers, and we...",51-9061,0.0625
3,3,http://data.europa.eu/esco/occupation/0022f466...,air traffic safety technician,3155,17-3023.01,electronics engineering technicians,17-3023,NaN
4,4,http://data.europa.eu/esco/occupation/002da35b...,hospitality revenue manager,2431,13-1161.00,market research analysts and marketing special...,13-1161,NaN
...,...,...,...,...,...,...,...,...
2937,2937,http://data.europa.eu/esco/occupation/ff656b3a...,demographer,2120,15-2041.00,statisticians,15-2041,NaN
2938,2938,http://data.europa.eu/esco/occupation/ff8d4065...,sorter labourer,9612,51-9199.01,recycling and reclamation workers,51-9199,1.0000
2939,2939,http://data.europa.eu/esco/occupation/ffa4dd5d...,armoured car guard,5414,33-9032.00,security guards,33-9032,NaN
2940,2940,http://data.europa.eu/esco/occupation/ffade2f4...,civil service administrative officer,2422,11-3011.00,administrative services managers,11-3011,NaN


Join GTP 2011 data to crosswalk at ONET 8-digit level

In [72]:
cw_onet_esco_merged = pd.merge(
    cw_onet_esco_merged,
    onet.green_occupations_gtp[["onet_code", "share_green_gtp"]],
    on="onet_code",
    how="left"
)

cw_onet_esco_merged

,id,concept_uri,preferred_label,isco_level_4,onet_code,onet_occupation,onet_code_6d,share_green_vona2018,share_green_gtp
0,0,http://data.europa.eu/esco/occupation/00030d09...,technical director,2166,27-1011.00,art directors,27-1011,NaN,NaN
1,1,http://data.europa.eu/esco/occupation/000e93a3...,metal drawing machine operator,8121,51-4021.00,"extruding and drawing machine setters, operato...",51-4021,NaN,NaN
2,2,http://data.europa.eu/esco/occupation/0019b951...,precision device inspector,7543,51-9061.00,"inspectors, testers, sorters, samplers, and we...",51-9061,0.0625,0.0625
3,3,http://data.europa.eu/esco/occupation/0022f466...,air traffic safety technician,3155,17-3023.01,electronics engineering technicians,17-3023,NaN,NaN
4,4,http://data.europa.eu/esco/occupation/002da35b...,hospitality revenue manager,2431,13-1161.00,market research analysts and marketing special...,13-1161,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2937,2937,http://data.europa.eu/esco/occupation/ff656b3a...,demographer,2120,15-2041.00,statisticians,15-2041,NaN,NaN
2938,2938,http://data.europa.eu/esco/occupation/ff8d4065...,sorter labourer,9612,51-9199.01,recycling and reclamation workers,51-9199,1.0000,1.0000
2939,2939,http://data.europa.eu/esco/occupation/ffa4dd5d...,armoured car guard,5414,33-9032.00,security guards,33-9032,NaN,NaN
2940,2940,http://data.europa.eu/esco/occupation/ffade2f4...,civil service administrative officer,2422,11-3011.00,administrative services managers,11-3011,NaN,NaN


**5) Join 6-digit ONET-level occupation metadata to IBS SOC 6D - ISCO 4D crosswalk**

In [111]:
cw_soc_isco = crosswalks.soc10_isco08_ibs
cw_soc_isco

,soc10,isco08
0,111011,1112
1,111011,1113
2,111011,1120
3,111021,1112
4,111021,1114
...,...,...
1126,553015,310
1127,553016,310
1128,553017,310
1129,553018,310


Join 6-digit level brown classification from Vona et al. 2018

In [96]:
brown_occupations_vona2018 = onet.brown_occupations_vona2018
brown_occupations_vona2018["soc_code"] = brown_occupations_vona2018["soc_code"].str.replace("-", "").astype(int)
brown_occupations_vona2018["is_brown_vona2018"] = np.ones(
    brown_occupations_vona2018.shape[0], dtype=int
)

In [102]:
cw_soc_isco_merged = pd.merge(
    cw_soc_isco,
    brown_occupations_vona2018.drop(columns=["occupation"]),
    left_on="soc10",
    right_on="soc_code",
    how="left"
).drop(columns=["soc_code"])

cw_soc_isco_merged["is_brown_vona2018"] = cw_soc_isco_merged["is_brown_vona2018"].fillna(0)

Join 6-digit level greeness scores from Vona et al. 2019

In [115]:
# TODO

Aggregate data by ISCO 4D level

In [136]:
onet_to_isco_4d = cw_soc_isco_merged.groupby("isco08")["is_brown_vona2018"].mean()
onet_to_isco_4d = onet_to_isco_4d.reset_index()
onet_to_isco_4d = onet_to_isco_4d.rename(columns={"is_brown_vona2018": "share_brown_vona2018"})
onet_to_isco_4d["isco08"] = onet_to_isco_4d["isco08"].astype(str)
onet_to_isco_4d

,isco08,share_brown_vona2018
0,110,0.0
1,210,0.0
2,310,0.0
3,1111,0.0
4,1112,0.0
...,...,...
431,9621,0.0
432,9622,0.0
433,9623,0.5
434,9624,0.0


**7) Combine all information into a single ESCO-level data set, attaching ONET data either via ESCO directly (ONET 8-digits), or via ISCO (SOC 6-digits)**

In [137]:
occ_master_ds = omd.copy()

Join 8-digit ONET data (via Nesta crosswalk)

In [138]:
occ_master_ds = pd.merge(
    occ_master_ds,
    cw_onet_esco_merged[["concept_uri", "share_green_gtp", "share_green_vona2018"]],
    left_on="conceptUri",
    right_on="concept_uri",
    how="left"
).drop(columns="concept_uri")

occ_master_ds["share_green_gtp"] = occ_master_ds["share_green_gtp"].fillna(0)
occ_master_ds["share_green_vona2018"] = occ_master_ds["share_green_vona2018"].fillna(0)
occ_master_ds

,conceptUri,preferredLabel,description,isco_level_4,isco_level_3,isco_level_2,isco_level_1,isco_label_4,isco_label_3,isco_label_2,isco_label_1,share_green_esco,share_brown_esco,share_neutral_esco,gbn_classification_esco,share_green_esco_ess,share_brown_esco_ess,share_neutral_esco_ess,gbn_classification_esco_ess,share_green_gtp,share_green_vona2018
0,http://data.europa.eu/esco/occupation/00030d09...,technical director,Technical directors realise the artistic visio...,2654,265,26,2,"Film, stage and related directors and producers",Creative and performing artists,"Legal, social and cultural professionals",Professionals,0.000000,0.0,1.000000,neutral,0.000000,0.0,1.000000,neutral,0.0,0.0
1,http://data.europa.eu/esco/occupation/1a7fb683...,video and motion picture director,Video and motion picture directors are respons...,2654,265,26,2,"Film, stage and related directors and producers",Creative and performing artists,"Legal, social and cultural professionals",Professionals,0.000000,0.0,1.000000,neutral,0.000000,0.0,1.000000,neutral,0.0,0.0
2,http://data.europa.eu/esco/occupation/2f372afe...,performance lighting director,Performance lighting directors determine what ...,2654,265,26,2,"Film, stage and related directors and producers",Creative and performing artists,"Legal, social and cultural professionals",Professionals,0.052632,0.0,0.947368,neutral,0.055556,0.0,0.944444,neutral,0.0,0.0
3,http://data.europa.eu/esco/occupation/30b25ee4...,animation director,Animation directors supervise and recruit mult...,2654,265,26,2,"Film, stage and related directors and producers",Creative and performing artists,"Legal, social and cultural professionals",Professionals,0.000000,0.0,1.000000,neutral,0.000000,0.0,1.000000,neutral,0.0,0.0
4,http://data.europa.eu/esco/occupation/3b6bea7d...,video and motion picture producer,Video and motion picture producers supervise t...,2654,265,26,2,"Film, stage and related directors and producers",Creative and performing artists,"Legal, social and cultural professionals",Professionals,0.000000,0.0,1.000000,neutral,0.000000,0.0,1.000000,neutral,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3003,http://data.europa.eu/esco/occupation/b059d331...,hawker,Hawkers sell goods and services on established...,9520,952,95,9,Street vendors (excluding food),Street vendors (excluding food),Street and related sales and service workers,Elementary occupations,0.000000,0.0,1.000000,neutral,0.000000,0.0,1.000000,neutral,0.0,0.0
3004,http://data.europa.eu/esco/occupation/96a0d8d3...,quick service restaurant crew member,"Quick service restaurant crew members prepare,...",9411,941,94,9,Fast food preparers,Food preparation assistants,Food preparation assistants,Elementary occupations,0.040000,0.0,0.960000,neutral,0.058824,0.0,0.941176,neutral,0.0,0.0
3005,http://data.europa.eu/esco/occupation/a02a1117...,pizzaiolo,Pizzaiolos are responsible for preparing and c...,9411,941,94,9,Fast food preparers,Food preparation assistants,Food preparation assistants,Elementary occupations,0.047619,0.0,0.952381,neutral,0.066667,0.0,0.933333,neutral,0.0,0.0
3006,http://data.europa.eu/esco/occupation/e1cf6897...,kitchen assistant,Kitchen assistants assist in the preparation o...,9412,941,94,9,Kitchen helpers,Food preparation assistants,Food preparation assistants,Elementary occupations,0.064516,0.0,0.935484,neutral,0.100000,0.0,0.900000,neutral,0.0,0.0


Join 6-digit ONET data (via ISCO 4D codes)

In [139]:
occ_master_ds = pd.merge(
    occ_master_ds,
    onet_to_isco_4d,
    left_on="isco_level_4",
    right_on="isco08",
    how="left"
).drop(columns=["isco08"])

In [140]:
occ_master_ds

,conceptUri,preferredLabel,description,isco_level_4,isco_level_3,isco_level_2,isco_level_1,isco_label_4,isco_label_3,isco_label_2,isco_label_1,share_green_esco,share_brown_esco,share_neutral_esco,gbn_classification_esco,share_green_esco_ess,share_brown_esco_ess,share_neutral_esco_ess,gbn_classification_esco_ess,share_green_gtp,share_green_vona2018,share_brown_vona2018
0,http://data.europa.eu/esco/occupation/00030d09...,technical director,Technical directors realise the artistic visio...,2654,265,26,2,"Film, stage and related directors and producers",Creative and performing artists,"Legal, social and cultural professionals",Professionals,0.000000,0.0,1.000000,neutral,0.000000,0.0,1.000000,neutral,0.0,0.0,0.0
1,http://data.europa.eu/esco/occupation/1a7fb683...,video and motion picture director,Video and motion picture directors are respons...,2654,265,26,2,"Film, stage and related directors and producers",Creative and performing artists,"Legal, social and cultural professionals",Professionals,0.000000,0.0,1.000000,neutral,0.000000,0.0,1.000000,neutral,0.0,0.0,0.0
2,http://data.europa.eu/esco/occupation/2f372afe...,performance lighting director,Performance lighting directors determine what ...,2654,265,26,2,"Film, stage and related directors and producers",Creative and performing artists,"Legal, social and cultural professionals",Professionals,0.052632,0.0,0.947368,neutral,0.055556,0.0,0.944444,neutral,0.0,0.0,0.0
3,http://data.europa.eu/esco/occupation/30b25ee4...,animation director,Animation directors supervise and recruit mult...,2654,265,26,2,"Film, stage and related directors and producers",Creative and performing artists,"Legal, social and cultural professionals",Professionals,0.000000,0.0,1.000000,neutral,0.000000,0.0,1.000000,neutral,0.0,0.0,0.0
4,http://data.europa.eu/esco/occupation/3b6bea7d...,video and motion picture producer,Video and motion picture producers supervise t...,2654,265,26,2,"Film, stage and related directors and producers",Creative and performing artists,"Legal, social and cultural professionals",Professionals,0.000000,0.0,1.000000,neutral,0.000000,0.0,1.000000,neutral,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3003,http://data.europa.eu/esco/occupation/b059d331...,hawker,Hawkers sell goods and services on established...,9520,952,95,9,Street vendors (excluding food),Street vendors (excluding food),Street and related sales and service workers,Elementary occupations,0.000000,0.0,1.000000,neutral,0.000000,0.0,1.000000,neutral,0.0,0.0,0.0
3004,http://data.europa.eu/esco/occupation/96a0d8d3...,quick service restaurant crew member,"Quick service restaurant crew members prepare,...",9411,941,94,9,Fast food preparers,Food preparation assistants,Food preparation assistants,Elementary occupations,0.040000,0.0,0.960000,neutral,0.058824,0.0,0.941176,neutral,0.0,0.0,0.0
3005,http://data.europa.eu/esco/occupation/a02a1117...,pizzaiolo,Pizzaiolos are responsible for preparing and c...,9411,941,94,9,Fast food preparers,Food preparation assistants,Food preparation assistants,Elementary occupations,0.047619,0.0,0.952381,neutral,0.066667,0.0,0.933333,neutral,0.0,0.0,0.0
3006,http://data.europa.eu/esco/occupation/e1cf6897...,kitchen assistant,Kitchen assistants assist in the preparation o...,9412,941,94,9,Kitchen helpers,Food preparation assistants,Food preparation assistants,Elementary occupations,0.064516,0.0,0.935484,neutral,0.100000,0.0,0.900000,neutral,0.0,0.0,0.0
